### DistilBERT Fine-Tuning for Insider Threat Detection
Fine-tunes a 3-class DistilBERT classifier on augmented insider threat text data.

Classes:
    0 = Normal
    1 = Job Search
    2 = Data Exfiltration

Pipeline:
    1. Load aggregated sessions dataset
    2. Extract text from 5 columns (email, HTTP URL, HTTP content, file names, file content)
    3. Assign 3-class labels using ground truth + keyword heuristics
    4. Augment minority classes via synonym replacement + back-translation
    5. Fine-tune distilbert-base-uncased for 5 epochs
    6. Save best model to SavedModels/final_insider_threat_model/

Output:
    SavedModels/final_insider_threat_model/ — tokenizer + model weights
    DatasetsProcessed/augmented_malicious_texts.json — cached augmented training data

Usage:
    Run this BEFORE MAS-Training-Clean.py.
    MAS-Training-Clean.py loads the saved model — no fine-tuning happens there.


### Cell 1: Imports

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup, MarianMTModel, MarianTokenizer
)
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import nlpaug.augmenter.word as naw
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
print("Imports complete")

[nltk_data] Downloading package wordnet to /home/dcis-dl-
[nltk_data]     pc-032/nltk_data...


Imports complete


### Cell 2: Configuration

In [2]:
CONFIG = {
    "dataset_path": "../DatasetsProcessed/data_exfiltration_sessions_aggregated_dataset.csv",
    "augmented_cache": "../DatasetsProcessed/augmented_malicious_texts.json",
    "model_save_dir": "../SavedModels/FineTunedLLMModel",
    "base_model": "distilbert-base-uncased",
    "num_labels": 3,       # Normal=0, Job Search=1, Data Exfiltration=2
    "max_length": 512,
    "batch_size": 16,
    "epochs": 5,
    "learning_rate": 2e-5,
    "seed": 42,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

# Augmentation settings
AUG_CONFIG = {
    "target_malicious": 300,       # Target augmented malicious count
    "syn_per_session": 2,          # Synonym replacement variants per session
    "syn_replace_pct": 0.15,       # % of words replaced per synonym pass
    "enable_back_translation": True,  # Set False to skip (slow, downloads ~600MB models)
    "bt_pivot_lang": "fr",         # Back-translate via French
    "benign_sample_size": 300,     # Benign samples for balanced training
    "job_search_target": 150,      # Target job search samples after augmentation
}

# Job search keywords for identifying class 1 sessions
JOB_KEYWORDS = [
    'linkedin', 'glassdoor', 'monster.com', 'indeed', 'resume',
    'careerbuilder', 'recruiter', 'taleo', 'interview', 'job search',
    'new position', 'salary negotiation', 'reference check', 'applying for'
]

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

print(f"Device: {CONFIG['device']}")
print("Configuration loaded")

Device: cuda
Configuration loaded


### Cell 3: Load Dataset and Build Texts

In [3]:
if os.path.exists(CONFIG['dataset_path']):
    print(f"Dataset Available: {CONFIG['dataset_path']}")
    print("Loading Dataset...")
    sessions_df = pd.read_csv(CONFIG['dataset_path'])

    # POST-LOAD FIXES
    # 1. Restore Datetime objects
    sessions_df['start'] = pd.to_datetime(sessions_df['start'])
    sessions_df['end']   = pd.to_datetime(sessions_df['end'])

    # 2. Handle Text Columns (NaNs become empty strings)
    text_cols = ['email_content_text', 'http_url_text', 'http_content_text', 'file_names_text', 'file_content_text']
    sessions_df[text_cols] = sessions_df[text_cols].fillna("")

else:
    raise FileNotFoundError(f"Dataset not found: {CONFIG['processed_file']}")

print(f"Dataset Shape: {sessions_df.shape}")
print(f"Malicious Sessions: {sessions_df['label'].sum()}")
sessions_df.head()

Dataset Available: ../DatasetsProcessed/data_exfiltration_sessions_aggregated_dataset.csv
Loading Dataset...
Dataset Shape: (1948933, 21)
Malicious Sessions: 47


,user,start,end,duration,is_weekend,is_after_hour,emails_count,ext_emails_count,attachments_count,total_email_size,...,http_count,http_url_text,http_content_text,cloud_uploads_count,usb_connects_count,file_copies_count,file_to_usb_count,file_names_text,file_content_text,label
0,AAB0162,2010-01-04 07:41:00,2010-01-04 15:41:00,28800.0,0,0,9,1,2,2615549,...,72,http://barnesandnoble.com/Joseph_Szigeti/hubay...,"One of those, Testosterone, was filmed in 2003...",0,0,0,0,,,0
1,AAB0162,2010-01-05 07:46:00,2010-01-05 15:46:00,28800.0,0,0,8,0,3,2837712,...,62,http://pcmag.com/Bill_Ponsford/cripes/1996_Hav...,"In late 1996, the Justice Department opened a ...",0,0,0,0,,,0
2,AAB0162,2010-01-06 07:45:00,2010-01-06 15:45:00,28800.0,0,0,9,0,2,3028297,...,72,http://chase.com/Ediacara_biota/ediacara/Senax...,"Sinnock died in May 1947, before finishing the...",0,0,0,0,,,0
3,AAB0162,2010-01-07 07:45:00,2010-01-07 15:45:00,28800.0,0,0,7,1,1,2095245,...,75,http://foxsports.com/Psittacosaurus/psittacosa...,"In 1917-18, she was fitted with better rangefi...",0,0,0,0,,,0
4,AAB0162,2010-01-08 07:50:00,2010-01-08 15:50:00,28800.0,0,0,9,2,0,238896,...,47,http://pnc.com/Magnetosphere_of_Jupiter/rj/Oev...,"DePeyster also sent out Joseph Ainsse, a local...",0,0,0,0,,,0


### Cell 4: Assign 3-Class Labels and Extract Texts

In [4]:
print("\nAssigning 3-class labels...")

def build_session_text(row):
    """Combine all 5 text columns into one string per session."""
    parts = []
    for col, prefix in [
        ('email_content_text', 'EMAIL'),
        ('http_url_text', 'URLS'),
        ('http_content_text', 'WEB'),
        ('file_names_text', 'FILES'),
        ('file_content_text', 'FILE_CONTENT'),
    ]:
        val = str(row.get(col, '')).strip()
        if val and val.lower() != 'nan':
            parts.append(f"{prefix}: {val}")
    return " | ".join(parts) if parts else ""

malicious_df = sessions_df[sessions_df['label'] == 1].copy()
benign_df = sessions_df[sessions_df['label'] == 0].copy()

# --- Class 2: Data Exfiltration (47 sessions) ---
exfil_texts = []
for _, row in malicious_df.iterrows():
    text = build_session_text(row)
    if text:
        exfil_texts.append(text)
print(f"  Class 2 (Exfiltration): {len(exfil_texts)} original sessions")

# --- Class 1: Job Search (keyword heuristic on benign sessions) ---
job_texts = []
for _, row in benign_df.iterrows():
    url_text = str(row.get('http_url_text', '')).lower()
    email_text = str(row.get('email_content_text', '')).lower()
    combined = url_text + " " + email_text
    if any(kw in combined for kw in JOB_KEYWORDS):
        text = build_session_text(row)
        if text:
            job_texts.append(text)
    if len(job_texts) >= 500:  # Cap to avoid processing too many
        break

print(f"  Class 1 (Job Search):   {len(job_texts)} sessions found via keywords")

# --- Class 0: Normal (sample from remaining benign) ---
normal_texts = []
for _, row in benign_df.sample(n=min(2000, len(benign_df)), random_state=42).iterrows():
    text = build_session_text(row)
    if text:
        normal_texts.append(text)
    if len(normal_texts) >= AUG_CONFIG["benign_sample_size"]:
        break

print(f"  Class 0 (Normal):       {len(normal_texts)} sessions sampled")


Assigning 3-class labels...
  Class 2 (Exfiltration): 46 original sessions
  Class 1 (Job Search):   500 sessions found via keywords
  Class 0 (Normal):       300 sessions sampled


### Cell 5: Text Augmentation - Synonym Replacement

In [5]:
print("\n" + "="*60)
print("  TIER 1: Synonym Replacement Augmentation")
print("="*60)

SYN_AVAILABLE = True

augmented_exfil = list(exfil_texts)  # Start with originals
augmented_jobs = list(job_texts)

if SYN_AVAILABLE:
    syn_aug = naw.SynonymAug(aug_src='wordnet', aug_p=AUG_CONFIG["syn_replace_pct"])
    
    # Augment exfiltration texts
    print(f"\nAugmenting {len(exfil_texts)} exfiltration texts "
          f"({AUG_CONFIG['syn_per_session']} variants each)...")
    for i, text in enumerate(exfil_texts):
        truncated = text[:2000]  # nlpaug is slow on very long texts
        try:
            variants = syn_aug.augment(truncated, n=AUG_CONFIG["syn_per_session"])
            if isinstance(variants, str):
                variants = [variants]
            augmented_exfil.extend(variants)
        except Exception as e:
            pass  # Skip if WordNet has no synonyms
        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(exfil_texts)}] done")
    
    print(f"  Exfiltration after synonym aug: {len(augmented_exfil)}")
    
    # Augment job search texts (if below target)
    if len(augmented_jobs) < AUG_CONFIG["job_search_target"]:
        needed = AUG_CONFIG["job_search_target"] - len(augmented_jobs)
        n_per = max(1, needed // len(job_texts)) if job_texts else 0
        print(f"\nAugmenting {len(job_texts)} job search texts ({n_per} variants each)...")
        for text in job_texts[:needed]:
            try:
                variants = syn_aug.augment(text[:2000], n=min(n_per, 2))
                if isinstance(variants, str):
                    variants = [variants]
                augmented_jobs.extend(variants)
            except Exception:
                pass
        print(f"  Job Search after synonym aug: {len(augmented_jobs)}")
else:
    # Fallback: simple duplication
    while len(augmented_exfil) < AUG_CONFIG["target_malicious"]:
        augmented_exfil.extend(exfil_texts)
    augmented_exfil = augmented_exfil[:AUG_CONFIG["target_malicious"]]

print(f"\nAfter Tier 1:")
print(f"  Exfiltration: {len(augmented_exfil)}")
print(f"  Job Search:   {len(augmented_jobs)}")
print(f"  Normal:        {len(normal_texts)}")


  TIER 1: Synonym Replacement Augmentation


ModuleNotFoundError: Missed nltk library. Install nltk by `pip install nltk`

### Cell 6: Text Augmentation - Back Translation

In [ ]:
if AUG_CONFIG["enable_back_translation"]:
    print("\n" + "="*60)
    print("  TIER 2: Back-Translation Augmentation")
    print("="*60)
    print("NOTE: This downloads ~600MB of translation models on first run.")
    
    try:
        pivot = AUG_CONFIG["bt_pivot_lang"]
        
        print(f"Loading EN→{pivot.upper()} model...")
        fwd_name = f'Helsinki-NLP/opus-mt-en-{pivot}'
        fwd_tok = MarianTokenizer.from_pretrained(fwd_name)
        fwd_model = MarianMTModel.from_pretrained(fwd_name)
        
        print(f"Loading {pivot.upper()}→EN model...")
        bwd_name = f'Helsinki-NLP/opus-mt-{pivot}-en'
        bwd_tok = MarianTokenizer.from_pretrained(bwd_name)
        bwd_model = MarianMTModel.from_pretrained(bwd_name)
        
        def back_translate(text, max_length=512):
            """Translate EN→pivot→EN."""
            tokens = fwd_tok([text[:max_length]], return_tensors='pt',
                             truncation=True, padding=True)
            fwd_out = fwd_model.generate(**tokens, max_length=max_length)
            pivot_text = fwd_tok.decode(fwd_out[0], skip_special_tokens=True)
            
            tokens = bwd_tok([pivot_text], return_tensors='pt',
                             truncation=True, padding=True)
            bwd_out = bwd_model.generate(**tokens, max_length=max_length)
            return bwd_tok.decode(bwd_out[0], skip_special_tokens=True)
        
        # Fill up to target count with back-translated samples
        remaining = AUG_CONFIG["target_malicious"] - len(augmented_exfil)
        if remaining > 0:
            print(f"\nBack-translating {remaining} more exfiltration texts...")
            source_texts = exfil_texts * ((remaining // len(exfil_texts)) + 1)
            for i, text in enumerate(source_texts[:remaining]):
                try:
                    bt_text = back_translate(text)
                    augmented_exfil.append(bt_text)
                except Exception:
                    pass
                if (i + 1) % 10 == 0:
                    print(f"  [{i+1}/{remaining}] done")
        
        print(f"\nAfter Tier 2 - Exfiltration: {len(augmented_exfil)}")
        
        # Clean up translation models to free GPU memory
        del fwd_model, bwd_model, fwd_tok, bwd_tok
        import gc; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    except Exception as e:
        print(f"Back-translation failed: {e}")
        print("Continuing with Tier 1 augmentation only.")
else:
    print("\nBack-translation disabled. Using Tier 1 only.")

# Trim to target counts
augmented_exfil = augmented_exfil[:AUG_CONFIG["target_malicious"]]
augmented_jobs = augmented_jobs[:AUG_CONFIG["job_search_target"]]

print(f"\nFinal augmented counts:")
print(f"  Class 0 (Normal):       {len(normal_texts)}")
print(f"  Class 1 (Job Search):   {len(augmented_jobs)}")
print(f"  Class 2 (Exfiltration): {len(augmented_exfil)}")

### Cell 7: Save Augmented Dataset Cache

In [ ]:
cache_records = []
for t in augmented_exfil:
    cache_records.append({"text": t, "label": 2, "class_name": "Data Exfiltration"})
for t in augmented_jobs:
    cache_records.append({"text": t, "label": 1, "class_name": "Job Search"})
for t in normal_texts:
    cache_records.append({"text": t, "label": 0, "class_name": "Normal"})

os.makedirs(os.path.dirname(CONFIG["augmented_cache"]), exist_ok=True)
with open(CONFIG["augmented_cache"], 'w') as f:
    json.dump(cache_records, f, indent=2)
print(f"\nAugmented dataset cached to {CONFIG['augmented_cache']}")
print(f"Total records: {len(cache_records)}")

# To reload in future runs:
# with open(CONFIG["augmented_cache"]) as f:
#     cache_records = json.load(f)

### Cell 8: Build Fine-Tuning Dataset

In [ ]:
all_texts = augmented_exfil + augmented_jobs + normal_texts
all_labels = ([2] * len(augmented_exfil) +
              [1] * len(augmented_jobs) +
              [0] * len(normal_texts))

# Shuffle
combined = list(zip(all_texts, all_labels))
random.shuffle(combined)
all_texts, all_labels = zip(*combined)
all_texts = list(all_texts)
all_labels = list(all_labels)

# Train/val split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    all_texts, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

print(f"\nFine-tuning split:")
print(f"  Train: {len(train_texts)} samples")
print(f"  Val:   {len(val_texts)} samples")
for cls in [0, 1, 2]:
    n_train = sum(1 for l in train_labels if l == cls)
    n_val = sum(1 for l in val_labels if l == cls)
    print(f"    Class {cls}: {n_train} train, {n_val} val")


class ThreatTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], return_tensors='pt', padding='max_length',
            truncation=True, max_length=self.max_length
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }


tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

train_dataset = ThreatTextDataset(train_texts, train_labels, tokenizer, CONFIG["max_length"])
val_dataset = ThreatTextDataset(val_texts, val_labels, tokenizer, CONFIG["max_length"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"])

print("\nDatasets and dataloaders ready.")

### Cell 9: Fine-Tune DistilBERT

In [ ]:
print("\n" + "="*60)
print("  Fine-Tuning DistilBERT")
print("="*60)

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["base_model"], num_labels=CONFIG["num_labels"]
)
model.to(CONFIG["device"])

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=0.01)
total_steps = len(train_loader) * CONFIG["epochs"]
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

best_val_loss = float('inf')

for epoch in range(CONFIG["epochs"]):
    # --- Training ---
    model.train()
    total_train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']} [Train]"):
        input_ids = batch['input_ids'].to(CONFIG["device"])
        attention_mask = batch['attention_mask'].to(CONFIG["device"])
        labels = batch['labels'].to(CONFIG["device"])

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    total_val_loss = 0
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(CONFIG["device"])
            attention_mask = batch['attention_mask'].to(CONFIG["device"])
            labels = batch['labels'].to(CONFIG["device"])

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()

            probs = torch.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"\n  Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # Save best model
        os.makedirs(CONFIG["model_save_dir"], exist_ok=True)
        model.save_pretrained(CONFIG["model_save_dir"])
        tokenizer.save_pretrained(CONFIG["model_save_dir"])
        print(f"  ✓ Best model saved (val_loss={best_val_loss:.4f})")

### Cell 10: Final Evaluation

In [ ]:
print("\n" + "="*60)
print("  Final Validation Evaluation")
print("="*60)

# Reload best model
model = AutoModelForSequenceClassification.from_pretrained(CONFIG["model_save_dir"])
model.to(CONFIG["device"])
model.eval()

all_preds, all_true, all_probs = [], [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(CONFIG["device"])
        attention_mask = batch['attention_mask'].to(CONFIG["device"])
        labels = batch['labels'].to(CONFIG["device"])

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

label_names = ["Normal", "Job Search", "Data Exfiltration"]
print(classification_report(all_true, all_preds, target_names=label_names))

# Threat detection AUC (binary: normal vs any threat)
binary_true = [1 if l > 0 else 0 for l in all_true]
binary_probs = [1.0 - p[0] for p in all_probs]  # P(threat) = 1 - P(normal)
try:
    auc = roc_auc_score(binary_true, binary_probs)
    print(f"Binary Threat Detection AUC: {auc:.4f}")
except ValueError:
    print("Could not compute AUC (only one class present in validation)")

print(f"\nModel saved to: {CONFIG['model_save_dir']}")
print("Done! This model will be loaded in MAS-Training.ipynb")